In [1]:
# Данный ноутбук использовал окружение google-colab
%pip install catboost fasttext datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.6/221.6 kB 21.8 MB

# Домашнее задание "NLP. Часть 1"

In [2]:
import math
import re
import os
import random
import json
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Any

import torch
import numpy as np
import datasets
import fasttext
import fasttext.util
from transformers import BertTokenizer, BertModel

/usr/local/lib/python3.12/dist-packages/torch_xla/experimental/gru.py:113: SyntaxWarning: invalid escape sequence '\_'
  * **h_n**: tensor of shape :math:`(D * \text{num\_layers}, H_{out})` or
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [3]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


seed_everything(42)

In [4]:
def normalize_pretokenize_text(text: str) -> List[str]:
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)
    return words

In [5]:
# This block is for tests only
test_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "never jump over the lazy dog quickly",
    "brown foxes are quick and dogs are lazy"
]


def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
    all_words = []
    for text in texts:
        words = normalize_pretokenize_text(text)
        all_words.extend(words)
    vocab = sorted(set(all_words))
    vocab_index = {word: idx for idx, word in enumerate(vocab)}
    return vocab, vocab_index


vocab, vocab_index = build_vocab(test_corpus)

## Задание 1 (0.5 балла)
Реализовать One-Hot векторизацию текстов

In [6]:
def one_hot_vectorization(
    text: str,
    vocab: List[str] = None,
    vocab_index: Dict[str, int] = None
) -> List[List[int]]:

    expected_len = len(vocab_index) if vocab is None else len(vocab)
    words = text.split()
    res = []

    for word in words:
        word_ohe = np.zeros(expected_len)

        if word in vocab_index:
            word_ohe[vocab_index[word]] = 1
        elif vocab is not None and word in vocab:
            word_ohe[vocab.index(word)] = 1

        res.append(word_ohe.tolist())

    return res


def test_one_hot_vectorization(
    vocab: List[str],
    vocab_index: Dict[str, int]
) -> bool:
    try:
        text = "the quick brown fox"
        result = one_hot_vectorization(text, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result[0]) != expected_length:
            return False

        words_in_text = normalize_pretokenize_text(text)
        for i, word in enumerate(words_in_text):
            if word in vocab_index:
                idx = vocab_index[word]
                if result[i][idx] != 1:
                    return False

        print("One-Hot-Vectors test PASSED")

        return True
    except Exception as e:
        print(f"One-Hot-Vectors test FAILED: {e}")
        return False

In [7]:
assert test_one_hot_vectorization(vocab, vocab_index)

One-Hot-Vectors test PASSED


## Задание 2 (0.5 балла)
Реализовать Bag-of-Words

In [8]:
from collections import Counter


def bag_of_words_vectorization(text: str) -> Dict[str, int]:

    return Counter(text.split())


def test_bag_of_words_vectorization() -> bool:
    try:
        text = "the the quick brown brown brown"
        result = bag_of_words_vectorization(text)

        if not isinstance(result, dict):
            return False

        if result.get('the', 0) != 2:
            return False
        if result.get('quick', 0) != 1:
            return False
        if result.get('brown', 0) != 3:
            return False
        if result.get('nonexistent', 0) != 0:
            return False

        print("Bad-of-Words test PASSED")
        return True
    except Exception as e:
        print(f"Bag-of-Words test FAILED: {e}")
        return False

In [9]:
assert test_bag_of_words_vectorization()

Bad-of-Words test PASSED


## Задание 3 (0.5 балла)
Реализовать TF-IDF

In [10]:
def tf_idf_vectorization(text: str, corpus: List[str] = None, vocab: List[str] = None, vocab_index: Dict[str, int] = None) -> List[float]:

    result_vector = [0.0] * len(vocab)
    words = text.split()
    word_counts = Counter(words)
    total_words = len(words)

    tf_values = {}
    for word, count in word_counts.items():
        if word in vocab_index:
            tf_values[word] = count / total_words

    idf_values = {}
    if corpus is not None:
        total_docs = len(corpus)

        doc_frequency = {word: 0 for word in vocab}
        for doc in corpus:
            doc_words = set(doc.split())
            for word in vocab:
                if word in doc_words:
                    doc_frequency[word] += 1

        for word in vocab:
            doc_freq = doc_frequency[word]
            if doc_freq > 0:
                idf_values[word] = math.log(total_docs / doc_freq)
            else:
                idf_values[word] = 0.0
    else:
        for word in vocab:
            idf_values[word] = 1.0

    for word, idx in vocab_index.items():
        tf = tf_values.get(word, 0.0)
        idf = idf_values.get(word, 0.0)
        result_vector[idx] = tf * idf

    return result_vector


def test_tf_idf_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "the quick brown"
        result = tf_idf_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("TF-IDF test PASSED")
        return True
    except Exception as e:
        print(f"TF-IDF test FAILED: {e}")
        return False

In [11]:
assert test_tf_idf_vectorization(test_corpus, vocab, vocab_index)

TF-IDF test PASSED


## Задание 4 (1 балл)
Реализовать Positive Pointwise Mutual Information (PPMI).  
https://en.wikipedia.org/wiki/Pointwise_mutual_information
$$PPMI(word, context) = max(0, PMI(word, context))$$
$$PMI(word, context) = log \frac{P(word, context)}{P(word) P(context)} = log \frac{N(word, context)|(word, context)|}{N(word) N(context)}$$
где $N(word, context)$ -- число вхождений слова $word$ в окно $context$ (размер окна -- гиперпараметр)

In [12]:
def ppmi_vectorization(
    text: str,
    corpus: List[str] = None,
    vocab: List[str] = None,
    vocab_index: Dict[str, int] = None,
    window_size: int = 2
) -> List[float]:

    word_counts = defaultdict(int)
    cooccurrence_counts = defaultdict(int)
    total_word_count = 0
    total_cooccurrence_count = 0

    for doc in corpus:
        words = doc.lower().split()

        for i, word in enumerate(words):
            if word in vocab_index:
                word_counts[word] += 1
                total_word_count += 1

                start = max(0, i - window_size)
                end = min(len(words), i + window_size + 1)

                for j in range(start, end):
                    if j != i and words[j] in vocab_index:
                        pair = tuple(sorted([word, words[j]]))
                        cooccurrence_counts[pair] += 1
                        total_cooccurrence_count += 1

    text_words = text.lower().split()
    text_vocab = [word for word in text_words if word in vocab_index]

    ppmi_vector = [0.0] * len(vocab)

    for i, vocab_word in enumerate(vocab):
        if word_counts[vocab_word] == 0:
            continue

        p_vocab_word = word_counts[vocab_word] / total_word_count

        ppmi_sum = 0.0
        context_count = 0

        for text_word in text_vocab:
            if word_counts[text_word] == 0:
                continue

            p_text_word = word_counts[text_word] / total_word_count

            pair = tuple(sorted([vocab_word, text_word]))
            if pair in cooccurrence_counts:
                p_joint = cooccurrence_counts[pair] / total_cooccurrence_count

                pmi = math.log(p_joint / (p_vocab_word *
                               p_text_word)) if p_joint > 0 else 0.0

                ppmi = max(0.0, pmi)
                ppmi_sum += ppmi
                context_count += 1

        if context_count > 0:
            ppmi_vector[i] = ppmi_sum / context_count

    return ppmi_vector


def test_ppmi_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "quick brown fox"
        result = ppmi_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("PPMI test PASSED")
        return True
    except Exception as e:
        print(f"PPMI test FAILED: {e}")
        return False

In [13]:
assert test_ppmi_vectorization(test_corpus, vocab, vocab_index)

PPMI test PASSED


## Задание 5 (1 балл)
Реализовать получение эмбеддингов из fasttext и bert (для bert лучше использовать CLS токен)

In [14]:
def get_fasttext_embeddings(text: str, model_path: str = None, model: any = None) -> List[np.ndarray]:
    if model is None:
        if model_path is None:
            fasttext.util.download_model('en', if_exists='ignore')
            model_path = 'cc.en.300.bin'
        model = fasttext.load_model(model_path)

    words = text.split()
    embeddings = []

    for word in words:
        word_vector = model.get_word_vector(word)
        embeddings.append(word_vector)

    return embeddings

In [15]:
text = 'quick brown fox'

get_fasttext_embeddings(text)

[array([ 0.02726553, -0.10176626, -0.00404413,  0.09996106, -0.01571898,
         0.04248569,  0.2073496 ,  0.0087942 , -0.03639914,  0.00647714,
         0.06384429,  0.07365575,  0.06052569,  0.09319542,  0.01747438,
         0.10222745,  0.04015287, -0.04139482,  0.04251796,  0.01454664,
        -0.03042826,  0.00366084, -0.02781409,  0.0133833 , -0.07660622,
        -0.02829335,  0.05362656,  0.01393907, -0.00974267,  0.01524175,
        -0.06859868, -0.01214927, -0.0870764 , -0.01884183,  0.04669119,
        -0.00323973, -0.01214865,  0.00729405, -0.01671829,  0.0223159 ,
        -0.03287907, -0.04704074,  0.00924372,  0.07564097,  0.02308894,
         0.22239473, -0.07354219,  0.00677742, -0.03897931, -0.0066923 ,
        -0.02401984, -0.00171614,  0.09846944,  0.01784112, -0.04556665,
         0.04260909, -0.03917933,  0.08568592, -0.05624855,  0.02018312,
        -0.00297635,  0.01065139, -0.04050962, -0.07184873,  0.09240421,
         0.05162618, -0.1122213 , -0.06686787, -0.0

In [16]:
def get_bert_embeddings(
    text: str,
    model_name: str = 'bert-base-uncased',
    pool_method: str = 'cls'
) -> np.ndarray:

    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name)

    model.eval()

    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=512,
        add_special_tokens=True
    )

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden_state = outputs.last_hidden_state

    embeddings = last_hidden_state[:, 0, :].squeeze().numpy()

    return embeddings

In [17]:
text = 'quick brown fox'

get_bert_embeddings(text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

array([-7.73346305e-01, -1.52948231e-01, -3.96935582e-01,  3.66077840e-01,
        9.68401656e-02,  2.99831390e-01, -1.80258051e-01,  4.50202763e-01,
       -6.11763954e-01, -8.78365040e-02, -1.57348692e-01,  1.83510453e-01,
        5.23930043e-02,  5.23251772e-01, -7.05118999e-02, -2.15942129e-01,
       -5.21603465e-01,  4.96029586e-01,  2.92460062e-02, -6.66671917e-02,
        2.59652644e-01,  6.21655658e-02, -4.24834430e-01, -2.75131494e-01,
        2.66273528e-01,  2.59303957e-01,  2.01499425e-02,  4.24166024e-02,
       -1.15835927e-01,  4.44841504e-01, -6.88870549e-02, -3.51457484e-02,
       -1.53644178e-02,  2.10702181e-01,  1.32001504e-01, -2.54508764e-01,
        3.67906034e-01, -4.36000340e-02, -1.01087928e-01, -8.18056520e-03,
        1.89968824e-01,  1.13654859e-01,  3.34827334e-01, -1.27592117e-01,
       -1.67597085e-01, -1.39888838e-01, -2.83615279e+00, -2.15618134e-01,
       -9.69090089e-02, -6.97880089e-01, -3.54389042e-01, -1.10328220e-01,
        2.13750765e-01,  

## Задание 6 (1.5 балла)
Реализовать обучение так, чтобы можно было поверх эмбеддингов, реализованных в предыдущих заданиях, обучить какую-то модель (вероятно неглубокую, например, CatBoost) на задаче классификации текстов ([IMDB](https://huggingface.co/datasets/stanfordnlp/imdb)).

In [28]:
def vectorize_dataset(
    dataset_name: str = "imdb",
    vectorizer_type: str = "bow",
    split: str = "train",
    sample_size: int = 2500
) -> Tuple[Any, List, List]:

    dataset = datasets.load_dataset(dataset_name, split=split)

    if sample_size:
        dataset = dataset.shuffle(seed=42)
        dataset = dataset.select(range(min(sample_size, len(dataset))))

    texts = [item['text']
             for item in dataset if 'text' in item and item['text'].strip()]
    labels = [item['label'] for item in dataset if 'label' in item]

    def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
        all_words = []
        for text in texts:
            words = normalize_pretokenize_text(text)
            all_words.extend(words)
        vocab = sorted(set(all_words))
        vocab_index = {word: idx for idx, word in enumerate(vocab)}
        return vocab, vocab_index

    vocab, vocab_index = build_vocab(texts)

    vectorized_data = []
    for text in texts:
        if vectorizer_type == "one_hot":
            vectorized_data.append(
                one_hot_vectorization(text, vocab, vocab_index))
        elif vectorizer_type == "bow":
            bow_dict = bag_of_words_vectorization(text)
            vector = [bow_dict.get(word, 0) for word in vocab]
            vectorized_data.append(vector)
        elif vectorizer_type == "tfidf":
            vectorized_data.append(tf_idf_vectorization(
                text, texts, vocab, vocab_index))
        elif vectorizer_type == "ppmi":
            vectorized_data.append(ppmi_vectorization(
                text, texts, vocab, vocab_index))
        elif vectorizer_type == "fasttext":
            embeddings = get_fasttext_embeddings(text)
            if embeddings:
                avg_embedding = np.mean(embeddings, axis=0)
                vectorized_data.append(avg_embedding.tolist())
            else:
                vectorized_data.append([0] * 300)
        elif vectorizer_type == "bert":
            embedding = get_bert_embeddings(text)
            vectorized_data.append(embedding.tolist())
        else:
            raise ValueError(f"Unknown vectorizer type: {vectorizer_type}")

    return (vectorized_data, labels)


In [29]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score, KFold


def train(
    embeddings_method="bow",
    test_size=0.2,
    val_size=0.2,
    cv_folds=5
):
    X, y = vectorize_dataset("imdb", embeddings_method, "train")
    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=val_size,
        random_state=42
    )

    X_train = np.array(X_train)
    X_val = np.array(X_val)
    y_train = np.array(y_train)
    y_val = np.array(y_val)
    # print(f'Train: {X_train.shape, y_train.shape}, Val: {X_val.shape, y_val.shape}')

    model = CatBoostClassifier(
        verbose=False,
        random_state=42
    )

    model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)

    y_val_pred = model.predict(X_val)
    print(
        f'accuracy: {accuracy_score(y_val_pred, y_val)}, f1_score: {f1_score(y_val_pred, y_val, average='binary')}')
    print(f'classification report {classification_report(y_val_pred, y_val)}')

In [ ]:
for embeddings_method in ["bow", "one_hot", "tfidf", "ppmi", "fasttext"]:
    train(embeddings_method=embeddings_method)

accuracy: 0.79, f1_score: 0.7969052224371374
classification report               precision    recall  f1-score   support

           0       0.74      0.84      0.78       226
           1       0.85      0.75      0.80       274

    accuracy                           0.79       500
   macro avg       0.79      0.79      0.79       500
weighted avg       0.80      0.79      0.79       500

